# 01 - Inference pipeline overview

This notebook confirms the **orchestration contract** of the inference pipeline: the order in which `InferencePipeline.run` sequences its stages, the dataclasses that carry state between them (`Run`, `Result`), the configuration surface (`InferenceConfig`), and the on-disk directory layout produced by `InferenceMetadata`.

Modules exercised: `pipelines.inference_pipeline.pipeline`, `pipelines.inference_pipeline.loader` (`InferenceMetadata`, `Run`), `pipelines.inference_pipeline.metrics` (`Result`), `configuration.inference_config`.

No data mount or checkpoint is touched; only the static structure is inspected.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Configuration surface

`InferenceConfig` is a dataclass that holds every knob the pipeline reads. We construct one with a throwaway `run_directory` and list its fields and default values so the available controls are visible at a glance.

In [ ]:
from dataclasses import fields
from configuration.inference_config import InferenceConfig, InferencePaths

cfg = InferenceConfig(run_directory=Path('/tmp/example_run'))

for f in fields(cfg):
    value = getattr(cfg, f.name)
    print(f'{f.name:22s} = {value!r}')


## Directory layout from `InferenceMetadata`

`InferenceMetadata` derives every output path from the config. We build it for a fixed `output_subdir` (so the timestamp branch is bypassed) and print the resulting tree relative to the run directory. `create_dirs` is intentionally **not** called here, so nothing is written to disk.

In [ ]:
from pipelines.inference_pipeline.loader import InferenceMetadata

cfg_named = InferenceConfig(run_directory=Path('/tmp/example_run'), output_subdir='demo')
meta      = InferenceMetadata(cfg_named)

paths = {
    'output_dir'    : meta.output_dir,
    'figures_dir'   : meta.figures_dir,
    'animations_dir': meta.animations_dir,
    'logs_dir'      : meta.logs_dir,
    'cube_dir'      : meta.cube_dir,
    'metrics_path'  : meta.metrics_path,
    'report_path'   : meta.report_path,
}

for name, p in paths.items():
    print(f'{name:15s} -> {p.relative_to(cfg_named.run_directory)}')

print()
print('figure_path helper:', meta.figure_path('pixel_mse').relative_to(cfg_named.run_directory))


## Stage sequence of `InferencePipeline.run`

The orchestrator delegates to private stage methods in a fixed order. We read the method's own source to display that order, which is the ground truth for how `Run` and `Result` flow through the pipeline.

In [ ]:
import inspect
from pipelines.inference_pipeline.pipeline import InferencePipeline

stage_methods = [
    '_setup', '_load_run', '_predict', '_compute_slice_indices',
    '_evaluate_metrics', '_build_report',
]

for name in stage_methods:
    method = getattr(InferencePipeline, name)
    doc    = (method.__doc__ or '').strip().splitlines()
    print(f'{name:24s} signature: {inspect.signature(method)}')


## State-carrying dataclasses

Two dataclasses move data between stages. `Run` is the loaded model plus dataset context produced by `RunLoader`; `Result` is the stitched prediction cube plus per-pixel metric maps produced by `Predictor`. We list their fields so the downstream notebooks can build synthetic instances against the exact schema.

In [ ]:
from dataclasses import fields as dc_fields
from pipelines.inference_pipeline.loader  import Run
from pipelines.inference_pipeline.metrics import Result

print('Run fields')
for f in dc_fields(Run):
    print(f'    {f.name:18s}: {f.type}')

print()
print('Result fields')
for f in dc_fields(Result):
    print(f'    {f.name:20s}: {f.type}')


## Visual summary of the stage flow

A small annotated diagram makes the linear dependency chain explicit: each box is a stage, each arrow the artifact handed forward.

In [ ]:
stages = [
    ('_setup',           'meta, logger, plotter'),
    ('_load_run',        'Run'),
    ('_predict',         'Result'),
    ('_evaluate_metrics','global_metrics dict'),
    ('compose / animate','figure_paths, gif_paths'),
    ('_build_report',    'report.md path'),
]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.axis('off')
y = np.linspace(0.9, 0.1, len(stages))

for i, (stage, artifact) in enumerate(stages):
    ax.add_patch(plt.Rectangle((0.05, y[i] - 0.05), 0.42, 0.09,
                               fc='C0', alpha=0.18, ec='C0'))
    ax.text(0.26, y[i], stage, ha='center', va='center', fontsize=10, family='monospace')
    ax.text(0.55, y[i], '-> ' + artifact, ha='left', va='center', fontsize=9, color='0.25')
    if i < len(stages) - 1:
        ax.annotate('', xy=(0.26, y[i + 1] + 0.045), xytext=(0.26, y[i] - 0.05),
                    arrowprops=dict(arrowstyle='->', color='0.4'))

ax.set_title('InferencePipeline.run stage flow')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## Expected visual outcome

The printed field listings should match the dataclass definitions exactly, the derived paths should all nest under `example_run/inference/demo/`, and the flow diagram should read top-to-bottom as setup -> load -> predict -> metrics -> figures -> report, confirming the orchestration order without running any heavy stage.